In [5]:
from _utils import *


# Configure logging
log_dir = "../logs"
if not os.path.exists(log_dir):
    os.makedirs(log_dir)

log_file = os.path.join(log_dir, f"model_db_download_errors_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log")
logging.basicConfig(
    filename=log_file,
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)

# Load dataset from Excel
df = pd.read_excel("../annotations/model_db_annotations.xlsx")

# Define output directories and files
data_dir = "../data/raw"
output_json = os.path.join(data_dir, "model_db_metadata.json")

# Create data dictionary to document the data structure
data_dictionary = {
    "dataset_description": "Metadata collection of neuron models from ModelDB including download URLs and file information",
    "created_date": datetime.now().strftime("%Y-%m-%d"),
    "source": "ModelDB - A database of computational neuroscience models",
    "fields": {
        "row_id": "Original row identifier from the annotations spreadsheet",
        "file_hash": "Hash value of the file content for identification",
        "raw_sha": "SHA hash of the raw file content",
        "count": "Occurrence count in the dataset",
        "url": "Original ModelDB URL for the file",
        "download_url": "Direct download URL constructed from the original URL",
        "filename": "Name of the file extracted from the URL",
        "content": "Raw content of the downloaded file when available",
        "error_code": "Error code if download failed, null for successful downloads",
        "download_date": "Timestamp when the metadata was recorded"
    },
    "entries": []  # Will store the actual data
}

# Failed downloads counter
failed_download_count = 0

# Create data directory if it doesn't exist
if not os.path.exists(data_dir):
    print(f"Creating directory: {data_dir}")
    os.makedirs(data_dir)

# Function to convert ModelDB URL to direct download URL
def get_direct_download_url(url):
    match = re.search(r"https://modeldb\.science/(\d+)\?tab=2&file=(.+)", url)
    if match:
        model_id, file_path = match.groups()
        return f"https://modeldb.science/getModelFile?model={model_id}&file={file_path}", file_path
    return None, None  # Return None if the URL doesn't match the expected pattern

# Function to fetch .mod file content and store in JSON
def fetch_mod_metadata(row_id, file_hash, raw_sha, count, url):
    global failed_download_count
    direct_url, file_path = get_direct_download_url(url)
    
    # Default structure for JSON entry
    file_entry = {
        "row_id": row_id,
        "file_hash": file_hash,
        "raw_sha": raw_sha,
        "count": count,
        "url": url,
        "download_url": direct_url,
        "filename": file_path,  # Store the filename separately
        "content": None,  # Default to None (indicating missing content)
        "error_code": None,  # New field for failed downloads
        "download_date": datetime.now().strftime("%Y-%m-%d %H:%M:%S")  # Add timestamp
    }
    
    if not direct_url:
        error_msg = f"Invalid URL format: {url}"
        print(f"Skipping invalid URL: {url}")
        logging.error(f"Row ID {row_id}: {error_msg}")
        file_entry["error_code"] = "Invalid URL"
        failed_download_count += 1
        data_dictionary["entries"].append(file_entry)  # Store even failed ones
        return
        
    try:
        response = requests.get(direct_url, timeout=10)
        response.raise_for_status()
        file_entry["content"] = response.text  # Store the downloaded content
    except requests.exceptions.HTTPError as http_err:
        error_msg = f"HTTP Error {response.status_code}: {http_err}"
        print(f"Failed to fetch {direct_url} - {error_msg}")
        logging.error(f"Row ID {row_id}, URL: {direct_url} - {error_msg}")
        file_entry["error_code"] = str(response.status_code)
        failed_download_count += 1
    except requests.exceptions.RequestException as e:
        error_msg = f"Request Error: {str(e)}"
        print(f"Failed to fetch {direct_url}: {e}")
        logging.error(f"Row ID {row_id}, URL: {direct_url} - {error_msg}")
        file_entry["error_code"] = "Request Error"
        failed_download_count += 1
        
    data_dictionary["entries"].append(file_entry)  # Store the file entry (success or failure)

# Select rows from the dataset
start = 0
n_rows = df.shape[0] 
df_subset = df.iloc[start:start+n_rows]

# Log the start of processing
logging.info(f"Starting metadata collection for {len(df_subset)} ModelDB entries")
print(f"Starting metadata collection for {len(df_subset)} ModelDB entries")

# Process selected files
for _, row in tqdm(df_subset.iterrows(), total=len(df_subset), desc="Collecting ModelDB metadata"):
    fetch_mod_metadata(
        row["row_id"], 
        row["file_hash"], 
        row["raw_sha"], 
        row["count"],  
        row["url"]
    )

# Add summary statistics to the data dictionary
data_dictionary["total_entries"] = len(data_dictionary["entries"])
data_dictionary["successful_downloads"] = len(data_dictionary["entries"]) - failed_download_count
data_dictionary["failed_downloads"] = failed_download_count

# Save the complete data dictionary to JSON
with open(output_json, "w", encoding="utf-8") as json_file:
    json.dump(data_dictionary, json_file, indent=4)

# Log completion
completion_message = f"\nModelDB metadata saved in {output_json}"
print(completion_message)
logging.info(completion_message)

stats_message = f"Total entries: {data_dictionary['total_entries']}, " \
               f"Successful downloads: {data_dictionary['successful_downloads']}, " \
               f"Failed downloads: {failed_download_count}"
print(stats_message)
logging.info(stats_message)

if failed_download_count > 0:
    print(f"Some metadata collections failed. Check log file: {log_file}")
    logging.info("Download process completed with errors")
else:
    logging.info("Download process completed successfully")

/home/imc33/.conda/envs/mod-annotation/lib/python3.10/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
/home/imc33/.conda/envs/mod-annotation/lib/python3.10/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed


Creating directory: ../data
Starting metadata collection for 5133 ModelDB entries


Failed to fetch https://modeldb.science/getModelFile?model=58173&file=b05oct26/napRT03.mod - HTTP Error 403: 403 Client Error: Forbidden for url: https://modeldb.science/getModelFile?model=58173&file=b05oct26/napRT03.mod


Failed to fetch https://modeldb.science/getModelFile?model=65218&file=nrntraub/mod/napf.mod - HTTP Error 403: 403 Client Error: Forbidden for url: https://modeldb.science/getModelFile?model=65218&file=nrntraub/mod/napf.mod


Failed to fetch https://modeldb.science/getModelFile?model=38241&file=CA1_multi/mechanism/not-currently-used/it1.mod - HTTP Error 403: 403 Client Error: Forbidden for url: https://modeldb.science/getModelFile?model=38241&file=CA1_multi/mechanism/not-currently-used/it1.mod


...

Failed to fetch https://modeldb.science/getModelFile?model=38229&file=kcum.mod - HTTP Error 403: 403 Client Error: Forbidden for url: https://modeldb.science/getModelFile?model=38229&file=kcum.mod


Failed to fetch https://modeldb.science/getModelFile?model=38235&file=CoHCNS2000/coh.mod - HTTP Error 403: 403 Client Error: Forbidden for url: https://modeldb.science/getModelFile?model=38235&file=CoHCNS2000/coh.mod


Failed to fetch https://modeldb.science/getModelFile?model=58173&file=b05oct26/cah.mod - HTTP Error 403: 403 Client Error: Forbidden for url: https://modeldb.science/getModelFile?model=58173&file=b05oct26/cah.mod


Failed to fetch https://modeldb.science/getModelFile?model=65218&file=nrntraub/mod/kdr_fs.mod - HTTP Error 403: 403 Client Error: Forbidden for url: https://modeldb.science/getModelFile?model=65218&file=nrntraub/mod/kdr_fs.mod



ModelDB metadata saved in ../data/model_db_metadata.json
Total entries: 5133, Successful downloads: 4951, Failed downloads: 182
Some metadata collections failed. Check log file: ../logs/model_db_download_errors_20251019_131449.log
